# SQuAD v2.0 — Extractive vs RAG Question Answering

## BITS Assignment Submission

This notebook contains the complete implementation of:
- **Pipeline A**: Extractive QA using `deepset/roberta-base-squad2`
- **Pipeline B**: RAG with TF-IDF retrieval and `flan-t5-base` generation

Both with from-scratch implementations of Recall@K, MRR, and MAP metrics.

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers datasets sentence-transformers scikit-learn numpy pandas matplotlib seaborn tqdm

## 2. Import Libraries

In [ ]:
import json
import time
import random
import re
import string
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer

print(f'PyTorch: {torch.__version__}')
print(f'Device: {\'cuda\' if torch.cuda.is_available() else \'cpu\'}')

## 3. Configuration

In [ ]:
# Configuration
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Dataset
DATASET_NAME = 'rajpurkar/squad_v2'
SAMPLE_SIZE = 100  # Change to 500 for full submission

# Models
EXTRACTIVE_MODEL = 'deepset/roberta-base-squad2'
GENERATIVE_MODEL = 'google/flan-t5-base'

print(f'Output dir: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')

## 4. Seed and Utilities

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)

print('Seed set to', SEED)

## 5. Metrics (from scratch)

In [ ]:
# Text normalization and scoring
_ARTICLES = re.compile(r'\b(a|an|the)\b', flags=re.UNICODE)
_PUNCT = set(string.punctuation)

def normalize_answer(text: str) -> str:
    if text is None:
        return ''
    text = text.lower()
    text = ''.join(ch for ch in text if ch not in _PUNCT)
    text = _ARTICLES.sub(' ', text)
    text = ' '.join(text.split())
    return text

def exact_match(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred: str, gold: str) -> float:
    p_tokens = normalize_answer(pred).split()
    g_tokens = normalize_answer(gold).split()
    if not p_tokens or not g_tokens:
        return float(p_tokens == g_tokens)
    common = Counter(p_tokens) & Counter(g_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p_tokens)
    recall = overlap / len(g_tokens)
    return 2 * precision * recall / (precision + recall)

def best_match_against_golds(pred: str, golds: List[str]) -> Tuple[int, float]:
    golds = list(golds) or ['']
    em = max(exact_match(pred, g) for g in golds)
    f1 = max(f1_score(pred, g) for g in golds)
    return em, f1

print('Scoring functions loaded')

In [ ]:
# Metrics from scratch
def reciprocal_rank(ranking: List[bool]) -> float:
    for i, is_relevant in enumerate(ranking, start=1):
        if is_relevant:
            return 1.0 / i
    return 0.0

def mean_reciprocal_rank(rankings: List[List[bool]]) -> float:
    return np.mean([reciprocal_rank(r) for r in rankings]) if rankings else 0.0

def recall_at_k(ranking: List[bool], k: int) -> float:
    n_relevant = sum(ranking)
    if n_relevant == 0:
        return 1.0
    n_relevant_in_top_k = sum(ranking[:k])
    return n_relevant_in_top_k / n_relevant

def average_precision(ranking: List[bool]) -> float:
    if sum(ranking) == 0:
        return 1.0
    score = 0.0
    num_relevant_so_far = 0
    for i, is_relevant in enumerate(ranking):
        if is_relevant:
            num_relevant_so_far += 1
            precision_at_i = num_relevant_so_far / (i + 1)
            score += precision_at_i
    return score / sum(ranking)

def mean_average_precision(rankings: List[List[bool]]) -> float:
    return np.mean([average_precision(r) for r in rankings]) if rankings else 0.0

print('Metrics loaded')

## 6. Data Loading

In [ ]:
@dataclass
class Example:
    qid: str
    question: str
    context: str
    answer_text: str
    is_impossible: bool
    title: str

def load_squad_v2(sample_size=100, seed=42):
    print(f'Loading {DATASET_NAME}...')
    dataset = load_dataset(DATASET_NAME, split='validation')
    
    examples = []
    for item in dataset:
        for qa in item['qas']:
            answers = qa.get('answers', [])
            answer_text = answers[0]['text'] if answers else ''
            
            example = Example(
                qid=qa['id'],
                question=qa['question'],
                context=item['context'],
                answer_text=answer_text,
                is_impossible=qa.get('is_impossible', False),
                title=item.get('title', ''),
            )
            examples.append(example)
    
    # Stratified sampling
    rng = np.random.RandomState(seed)
    answerables = [e for e in examples if not e.is_impossible]
    unanswerables = [e for e in examples if e.is_impossible]
    
    n_ans = int(sample_size * 0.5)
    n_unans = sample_size - n_ans
    
    sampled = (list(rng.choice(answerables, min(n_ans, len(answerables)), replace=False)) +
                list(rng.choice(unanswerables, min(n_unans, len(unanswerables)), replace=False)))
    rng.shuffle(sampled)
    
    return sampled

examples = load_squad_v2(SAMPLE_SIZE, SEED)
print(f'Loaded {len(examples)} examples')

In [ ]:
# Build corpus
def build_corpus(examples):
    doc_texts = []
    doc_ids = []
    qid_to_gold_doc = {}
    seen = set()
    
    for ex in examples:
        if ex.title not in seen:
            doc_ids.append(ex.title)
            doc_texts.append(ex.context)
            seen.add(ex.title)
        qid_to_gold_doc[ex.qid] = ex.title
    
    return doc_ids, doc_texts, qid_to_gold_doc

doc_ids, doc_texts, qid_to_gold_doc = build_corpus(examples)
print(f'Built corpus: {len(doc_ids)} docs')

## 7. Pipeline A: Extractive QA

In [ ]:
@dataclass
class ExtractiveQAPrediction:
    qid: str
    question: str
    answer: str
    score: float
    no_answer: bool

class ExtractiveQA:
    def __init__(self, model_name=EXTRACTIVE_MODEL, device=DEVICE):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device).eval()
        self.device = device
    
    def predict_batch(self, examples):
        predictions = []
        for ex in tqdm(examples, desc='Extractive QA'):
            inputs = self.tokenizer(ex.question, ex.context, max_length=512, 
                                   truncation='only_second', return_tensors='pt').to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            start_idx = torch.argmax(outputs.start_logits)
            end_idx = torch.argmax(outputs.end_logits) + 1
            
            answer_tokens = inputs.input_ids[0, start_idx:end_idx]
            answer = self.tokenizer.decode(answer_tokens, skip_special_tokens=True)
            score = float(torch.softmax(outputs.start_logits, dim=-1).max())
            
            pred = ExtractiveQAPrediction(
                qid=ex.qid,
                question=ex.question,
                answer=answer,
                score=score,
                no_answer=False
            )
            predictions.append(pred)
        
        return predictions

print('ExtractiveQA class defined')

In [ ]:
# Run Pipeline A
print('\n' + '='*60)
print('PIPELINE A: EXTRACTIVE QA')
print('='*60)

t0 = time.time()
ext = ExtractiveQA(device=DEVICE)
ext_preds = ext.predict_batch(examples)
print(f'Completed in {time.time()-t0:.1f}s')

# Evaluate
em_scores = []
f1_scores = []
for ex, pred in zip(examples, ext_preds):
    if not ex.is_impossible:
        em, f1 = best_match_against_golds(pred.answer, [ex.answer_text])
        em_scores.append(em)
        f1_scores.append(f1)

ext_metrics = {
    'em': float(np.mean(em_scores)) if em_scores else 0.0,
    'f1': float(np.mean(f1_scores)) if f1_scores else 0.0,
}

print(f"EM: {ext_metrics['em']:.3f}")
print(f"F1: {ext_metrics['f1']:.3f}")
save_json(ext_metrics, OUTPUT_DIR / 'extractive_metrics.json')

## 8. Pipeline B: RAG

In [ ]:
class TFIDFRetriever:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        self.doc_ids = None
        self.doc_tfidf = None
    
    def fit(self, doc_ids, doc_texts):
        self.doc_ids = doc_ids
        self.doc_tfidf = self.vectorizer.fit_transform(doc_texts)
        return self
    
    def retrieve(self, query, k=5):
        query_vec = self.vectorizer.transform([query])
        scores = query_vec.dot(self.doc_tfidf.T).toarray()[0]
        top_k_indices = np.argsort(-scores)[:k]
        return [self.doc_ids[i] for i in top_k_indices if scores[i] > 0]

@dataclass
class RAGPrediction:
    qid: str
    question: str
    answer: str
    retrieved_docs: List[str]

print('RAG classes defined')

In [ ]:
# Run Pipeline B
print('\n' + '='*60)
print('PIPELINE B: RAG')
print('='*60)

t0 = time.time()
print('Fitting retriever...')
retriever = TFIDFRetriever().fit(doc_ids, doc_texts)
print(f'Retriever fitted in {time.time()-t0:.1f}s')

print('Loading generator...')
tokenizer = AutoTokenizer.from_pretrained(GENERATIVE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(GENERATIVE_MODEL).to(DEVICE).eval()

rag_preds = []
for ex in tqdm(examples, desc='RAG generation'):
    retrieved = retriever.retrieve(ex.question, k=5)
    context = ' '.join(retrieved) if retrieved else 'No context'
    
    prompt = f'Context: {context} Question: {ex.question} Answer:'
    inputs = tokenizer(prompt, max_length=512, truncation=True, return_tensors='pt').to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(inputs['input_ids'], max_new_tokens=30, num_beams=4, do_sample=False)
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    pred = RAGPrediction(
        qid=ex.qid,
        question=ex.question,
        answer=answer,
        retrieved_docs=retrieved
    )
    rag_preds.append(pred)

print(f'RAG completed')

In [ ]:
# Evaluate RAG
em_scores = []
f1_scores = []
for ex, pred in zip(examples, rag_preds):
    if not ex.is_impossible:
        em, f1 = best_match_against_golds(pred.answer, [ex.answer_text])
        em_scores.append(em)
        f1_scores.append(f1)

rag_metrics = {
    'em': float(np.mean(em_scores)) if em_scores else 0.0,
    'f1': float(np.mean(f1_scores)) if f1_scores else 0.0,
}

print(f"EM: {rag_metrics['em']:.3f}")
print(f"F1: {rag_metrics['f1']:.3f}")
save_json(rag_metrics, OUTPUT_DIR / 'rag_metrics.json')

## 9. Comparison and Results

In [ ]:
# Create comparison table
comparison_data = []
for ex, ext_pred, rag_pred in zip(examples[:10], ext_preds[:10], rag_preds[:10]):
    comparison_data.append({
        'Question': ex.question[:50],
        'Gold': ex.answer_text[:30],
        'Extractive': ext_pred.answer[:30],
        'RAG': rag_pred.answer[:30],
    })

df = pd.DataFrame(comparison_data)
print('\nSample Predictions:')
print(df.to_string())

df.to_csv(OUTPUT_DIR / 'sample_predictions.csv', index=False)

In [ ]:
# Final Summary
summary = {
    'dataset': DATASET_NAME,
    'sample_size': len(examples),
    'extractive_em': ext_metrics['em'],
    'extractive_f1': ext_metrics['f1'],
    'rag_em': rag_metrics['em'],
    'rag_f1': rag_metrics['f1'],
}

save_json(summary, OUTPUT_DIR / 'summary.json')

print('\n' + '='*60)
print('FINAL RESULTS')
print('='*60)
print(f"Extractive QA - EM: {ext_metrics['em']:.3f}, F1: {ext_metrics['f1']:.3f}")
print(f"RAG - EM: {rag_metrics['em']:.3f}, F1: {rag_metrics['f1']:.3f}")
print(f'\nAll outputs saved to: {OUTPUT_DIR}')
print('='*60)